[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C53_RealTime_Detectors_Course/03_rtmdet/03_rtmdet.ipynb)

# 03 · RTMDet 解剖（有效感受野 / 5×5 depthwise / 共享头 / 软标签 / 模型缩放）

目标：把 RTMDet 的四个核心设计**用数字验证一遍**，而不是背结构图。

本 notebook 你会亲手实现：

1. **理论感受野**（TRF）的递推计算器（含 stride 与 dilation）
2. **有效感受野**（ERF）的**梯度回传数值实验** —— 从中心输出单元反传，测出真实的高斯钟形
3. 验证解析公式 $\sigma_{ERF}=\sqrt{\sum d_\ell^2(k_\ell^2-1)/12}$ —— **实测与理论精确到小数点后 8 位**
4. **两个 3×3 vs 一个 5×5**：同样的 TRF，ERF 差 $\sqrt{1.5}=1.2247$ 倍
5. **ReLU 门控**让 ERF 进一步缩小的量化（有效面积 205 → 52）
6. 大核的**成本账**：depthwise/pointwise 拆分、算术强度、固定预算下的最优核尺寸
7. **共享 conv + 每层独立 BN**：参数量账 + 共享 BN 会把统计量污染成什么样
8. **DynamicSoftLabelAssigner 完整代价矩阵**（软标签 + IoU + 软中心先验）
9. **模型缩放计算器**：tiny→x 的参数量/FLOPs，与官方数字对比

> 心智模型：**理论感受野是几何量，有效感受野是统计量。
> 精度跟着后者走，而后者按 √N 增长 —— 所以「加大单层核」比「多堆几层」划算。**

## 1 · 理论感受野：一个纯几何量（与权重无关）

递推：$r_\ell = r_{\ell-1} + (k_\ell-1)\cdot d_\ell \cdot j_{\ell-1}$，$j_\ell = j_{\ell-1}\cdot s_\ell$，$r_0=1,\ j_0=1$。

先只看 stride=1、dilation=1 的情况：**TRF 直径 $= 1 + \sum_\ell (k_\ell - 1)$**。

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

def trf_stride1(kernels):
    '''stride=1, dilation=1 的堆叠：TRF 直径 = 1 + sum(k-1)。'''
    return 1 + sum(k - 1 for k in kernels)

print(f'{"配置":<22s} {"层数":>5s} {"TRF直径":>8s} {"TRF半径":>8s}')
CONFIGS = [
    ('3x3 x 1',      [3]),
    ('3x3 x 2',      [3, 3]),
    ('5x5 x 1',      [5]),
    ('3x3 x 10',     [3] * 10),
    ('5x5 x 10',     [5] * 10),
    ('7x7 x 10',     [7] * 10),
    ('3x3 x 64',     [3] * 64),
    ('CSPNeXtBlock', [3, 5, 1]),      # 3x3 + 5x5 depthwise + 1x1
]
for name, ks in CONFIGS:
    r = trf_stride1(ks)
    print(f'{name:<22s} {len(ks):>5d} {r:>8d} {r//2:>8d}')

# 两个 3x3 与一个 5x5 的 TRF **完全相同** —— 这是「小核堆叠等价大核」说法的来源
assert trf_stride1([3, 3]) == trf_stride1([5]) == 5
assert trf_stride1([3, 3, 3]) == trf_stride1([7]) == 7
assert trf_stride1([3] * 10) == 21
print()
print('✅ 两个 3x3 的 TRF 与一个 5x5 完全相同（都是 5x5）。')
print('   「小核堆叠可以等价替代大核」这句话，**在 TRF 意义上是对的**。')
print('⚠️  但精度并不跟着 TRF 走。下一节我们把真正起作用的那个量测出来。')

## 2 · 有效感受野：梯度回传法把它测出来

方法（Luo et al. NeurIPS 2016 的原始做法）：
**从输出特征图的中心单元反传一个单位梯度，看它在输入上留下什么形状。**

对线性卷积栈，反传就是「用 180° 旋转后的核再卷一次」；
我们用均匀核（每个位置权重 $1/k^2$），这样结果可以与解析式精确对拍。

In [ ]:
def conv_same(x, w, d=1):
    '''单通道 same 卷积，支持 dilation。'''
    k = w.shape[0]
    p = (k - 1) * d // 2
    xp = np.pad(x, p)
    out = np.zeros_like(x)
    H, W = x.shape
    for a in range(k):
        for b in range(k):
            out += w[a, b] * xp[a * d:a * d + H, b * d:b * d + W]
    return out

def erf_map(specs, size=161, gate=None, seed=0):
    '''梯度回传法测 ERF。specs = [(kernel, dilation), ...]（从输出侧往输入侧走）。
       均匀核是中心对称的，所以 180 度旋转后还是它自己 -> 反传就是再卷一次。
       gate=None 表示线性网络；gate=p 表示每层有 p 的概率保留梯度（模拟 ReLU 门控）。'''
    g = np.zeros((size, size))
    c = size // 2
    g[c, c] = 1.0                                   # ← 中心输出单元的单位梯度
    rg = np.random.default_rng(seed)
    for (k, d) in specs:
        g = conv_same(g, np.ones((k, k)) / (k * k), d)
        if gate is not None:
            g = g * (rg.random((size, size)) < gate)
    return g

def spatial_std(m):
    '''ERF 的空间标准差（把归一化后的梯度图当成一个二维概率分布）。'''
    s = m.sum()
    if s <= 0:
        return 0.0
    p = m / s
    n = m.shape[0]
    x = np.arange(n) - n // 2
    px = p.sum(0)
    mu = (px * x).sum()
    return float(np.sqrt((px * (x - mu) ** 2).sum()))

def show_erf(m, half=10, chars=' .:-=+*#%@'):
    '''把 ERF 中心区域画成字符热力图。'''
    c = m.shape[0] // 2
    sub = m[c - half:c + half + 1, c - half:c + half + 1]
    v = sub / max(sub.max(), 1e-30)
    for row in v:
        print('  ' + ''.join(chars[min(int(x * (len(chars) - 1) + 0.5), len(chars) - 1)]
                              for x in row))

E = erf_map([(3, 1)] * 10)
print('3x3 堆 10 层的 ERF（TRF 直径 21，下面画的是中心 21x21 区域）：')
show_erf(E, half=10)
print()
print(f'梯度总质量 = {E.sum():.6f}（线性网络下严格守恒 = 1）')
print(f'ERF 空间标准差 = {spatial_std(E):.6f}')
assert abs(E.sum() - 1.0) < 1e-9, '线性卷积栈的梯度质量守恒'
assert E[E.shape[0] // 2, E.shape[1] // 2] == E.max(), '中心贡献最大'
print()
print('⚠️  形状是**钟形而不是方块**：TRF 说「21x21 都可能有贡献」，')
print('    但边缘像素的梯度已经接近 0 —— 模型实际用得上的只有中间那一小块。')

## 3 · 解析式验证：$\sigma_{ERF}=\sqrt{\sum d_\ell^2 (k_\ell^2-1)/12}$

把每层归一化的卷积核看成一个概率分布，堆叠 = 分布自卷积。
宽度 $k$ 的离散均匀分布方差是 $(k^2-1)/12$；空洞率 $d$ 把坐标拉伸 $d$ 倍，方差乘 $d^2$。
**方差可加 ⇒ 标准差按 $\sqrt{N}$ 增长。这在均匀核线性网络下是精确等式，不是近似。**

In [ ]:
def erf_sigma_theory(specs):
    return float(np.sqrt(sum(d * d * (k * k - 1) / 12 for k, d in specs)))

print(f'{"配置":<24s} {"实测 sigma":>13s} {"理论 sigma":>13s} {"绝对误差":>11s}')
CASES = [
    ('3x3 x 6',                 [(3, 1)] * 6),
    ('5x5 x 6',                 [(5, 1)] * 6),
    ('7x7 x 6',                 [(7, 1)] * 6),
    ('7x7 dilation=3 x 1',      [(7, 3)]),
    ('混合 3,5,3(d2),5,3',      [(3, 1), (5, 1), (3, 2), (5, 1), (3, 1)]),
]
for name, sp in CASES:
    meas = spatial_std(erf_map(sp))
    theo = erf_sigma_theory(sp)
    print(f'{name:<24s} {meas:>13.8f} {theo:>13.8f} {abs(meas - theo):>11.2e}')
    assert abs(meas - theo) < 1e-9, (name, meas, theo)

print()
print('✅ 实测与理论在小数点后 8 位完全一致 —— 这不是拟合，是恒等式。')
print()
print('现在看 ERF/TRF 的比值随深度怎么衰减：')
print(f'{"配置":<16s} {"TRF半径":>8s} {"ERF sigma":>11s} {"ERF/TRF":>9s} {"1/sqrt(N) 参考":>15s}')
for k in [3, 5, 7]:
    for N in [1, 10]:
        sp = [(k, 1)] * N
        sig = erf_sigma_theory(sp)
        r = N * (k - 1) / 2
        print(f'{f"{k}x{k} x {N}":<16s} {r:>8.0f} {sig:>11.4f} {sig / r:>9.4f} '
              f'{1 / np.sqrt(N):>15.4f}')

for N in [1, 4, 16, 64]:
    sp = [(3, 1)] * N
    ratio = erf_sigma_theory(sp) / (N * 1.0)
    print(f'  3x3 x {N:<3d}  ERF/TRF = {ratio:.4f}')
r1 = erf_sigma_theory([(3, 1)] * 4) / 4
r2 = erf_sigma_theory([(3, 1)] * 64) / 64
assert abs(r1 / r2 - 4.0) < 1e-9, 'N 涨 16 倍，ERF/TRF 应恰好降到 1/4'
print()
print('✅ N 从 4 涨到 64（16 倍），ERF/TRF 恰好降到 1/4 = 1/sqrt(16)。')
print('⚠️  推论：**深网络的 TRF 早就覆盖全图，但 ERF 只有十几个像素。**')
print('    「感受野够大了」这句话如果指的是 TRF，基本没有信息量。')

## 4 · 两个 3×3 vs 一个 5×5：同样的 TRF，不同的 ERF

这是「为什么要用大核」最干净的一个证据。

In [ ]:
a = [(3, 1), (3, 1)]     # 两层 3x3
b = [(5, 1)]             # 一层 5x5
sa, sb = spatial_std(erf_map(a)), spatial_std(erf_map(b))

print(f'{"配置":<14s} {"层数":>5s} {"TRF直径":>8s} {"ERF sigma":>11s}')
print(f'{"3x3 x 2":<14s} {2:>5d} {trf_stride1([3,3]):>8d} {sa:>11.6f}')
print(f'{"5x5 x 1":<14s} {1:>5d} {trf_stride1([5]):>8d} {sb:>11.6f}')
print()
print(f'TRF 完全相同（都是 5），但 ERF 比值 = {sb / sa:.6f}')
print(f'理论值 sqrt( (24/12) / (2*8/12) ) = sqrt(1.5) = {np.sqrt(1.5):.6f}')

assert trf_stride1([3, 3]) == trf_stride1([5])
assert abs(sb / sa - np.sqrt(1.5)) < 1e-9
assert sb > sa
print()
print('✅ 用一层 5x5 换掉两层 3x3：')
print('   · TRF 不变')
print('   · 深度减半  -> **串行延迟减半、梯度路径变短**')
print(f'   · ERF 反而大 {(sb/sa-1)*100:.1f}%')
print()
print('同深度下的差距更大（RTMDet 是在**不减层**的前提下把 3x3 换成 5x5 depthwise）：')
print(f'{"N":>4s} {"3x3 sigma":>11s} {"5x5 sigma":>11s} {"比值":>8s}')
for N in [2, 4, 8, 16]:
    s3 = erf_sigma_theory([(3, 1)] * N)
    s5 = erf_sigma_theory([(5, 1)] * N)
    print(f'{N:>4d} {s3:>11.4f} {s5:>11.4f} {s5 / s3:>8.4f}')
assert abs(erf_sigma_theory([(5, 1)] * 8) / erf_sigma_theory([(3, 1)] * 8) - np.sqrt(3.0)) < 1e-9
print()
print(f'✅ 同深度下 5x5 相对 3x3 的 ERF 比值恒为 sqrt(24/8) = sqrt(3) = {np.sqrt(3):.4f}，与深度无关。')

## 5 · ReLU 门控：真实网络的 ERF 还要再小一圈

上面是线性网络。真实网络里 ReLU 会随机切断梯度路径。
用「每层以概率 $p$ 保留梯度」模拟，看两个量：
**梯度总质量**（衰减多快）与**有效面积** $(\sum m)^2/\sum m^2$（参与度，即「多少个像素在实质贡献」）。

In [ ]:
def participation(m):
    '''有效面积：全部质量集中在 1 个像素时 = 1，均匀摊在 n 个像素上时 = n。'''
    return float(m.sum() ** 2 / (m ** 2).sum())

print('5x5 堆 8 层，5 个随机种子取平均：')
print(f'{"保留概率":>9s} {"梯度总质量":>13s} {"有效面积":>10s} {"相对线性网络":>13s}')
res = {}
for gate in [None, 0.7, 0.5, 0.3]:
    ms, ps = [], []
    for s in range(5):
        m = erf_map([(5, 1)] * 8, size=121, gate=gate, seed=s)
        ms.append(m.sum())
        ps.append(participation(m))
    res[gate] = (float(np.mean(ms)), float(np.mean(ps)))
    tag = '线性（无门控）' if gate is None else f'p={gate}'
    print(f'{tag:>9s} {res[gate][0]:>13.3e} {res[gate][1]:>10.1f} '
          f'{res[gate][1] / res[None][1]:>13.2f}x')

assert res[0.7][1] < res[None][1]
assert res[0.5][1] < res[0.7][1]
assert res[0.3][1] < res[0.5][1], '门控越强，有效面积越小'
assert res[0.3][0] < res[None][0] * 1e-3, '梯度质量随门控指数衰减'
print()
print('单个种子（p=0.3）的 ERF 形状 —— 注意它变得**稀疏且不规则**：')
show_erf(erf_map([(5, 1)] * 8, size=121, gate=0.3, seed=3), half=10)
print()
print(f'⚠️  有效面积从 {res[None][1]:.0f} 掉到 {res[0.3][1]:.0f}（缩小 {res[None][1]/res[0.3][1]:.1f} 倍），')
print(f'    梯度总质量掉了 {np.log10(res[None][0]/res[0.3][0]):.0f} 个数量级。')
print('✅ 所以真实网络的 ERF **比线性估计还要小**。')
print('⚠️  但也别把结论推过头：Luo 的论文指出**训练后的 ERF 显著大于初始化时的 ERF**。')
print('    「ERF 不够大」有时是训练不充分的症状，不一定是结构问题。')

## 6 · 成本账：为什么 5×5 几乎免费，又为什么不是 11×11

In [ ]:
def conv_params(cin, cout, k, groups=1):
    return (cin // groups) * cout * k * k

def conv_macs(cin, cout, k, hw, groups=1):
    return conv_params(cin, cout, k, groups) * hw * hw

print('① 参数量：depthwise 让核尺寸变成零头')
print(f'{"C":>5s} {"5x5 DW":>10s} {"2x(3x3 dense)":>15s} {"倍数":>8s} '
      f'{"5x5DW 占 DW+PW 的":>18s}')
for C in [64, 128, 256, 512]:
    dw5 = conv_params(C, C, 5, groups=C)
    dense2 = 2 * conv_params(C, C, 3)
    pw = conv_params(C, C, 1)
    print(f'{C:>5d} {dw5:>10,d} {dense2:>15,d} {dense2 / dw5:>7.0f}x {dw5 / (dw5 + pw):>17.1%}')

C = 256
dw3, dw5 = conv_params(C, C, 3, groups=C), conv_params(C, C, 5, groups=C)
pw = conv_params(C, C, 1)
grow = (dw5 - dw3) / (dw3 + pw)
print()
print(f'C=256 时，把 block 里的 3x3 DW 换成 5x5 DW，整个 block 参数只涨 {grow:.1%}')
assert grow < 0.10, '5x5 相对 3x3 的参数增量应小于 10%'
assert dw5 / (dw5 + pw) < 0.10, 'C=256 时空间核只占可分离 block 的 9%'
print('✅ 「5x5 depthwise 几乎免费」的准确含义：相对同一个 block 里的 1x1 pointwise，')
print('   空间核便宜到可以忽略 —— **成本几乎全在 pointwise 上**。')

In [ ]:
print('② 算术强度（MAC 数 / 访存元素数）：depthwise 是**访存受限**的')
HW = 40
print(f'{"算子 (C=256, 40x40)":<26s} {"MACs":>14s} {"访存元素":>11s} {"算术强度":>10s} {"瓶颈":>8s}')
rows = []
for name, k, grp in [('dense 3x3', 3, 1), ('dense 1x1', 1, 1),
                     ('depthwise 3x3', 3, 256), ('depthwise 5x5', 5, 256),
                     ('depthwise 7x7', 7, 256), ('depthwise 11x11', 11, 256)]:
    macs = conv_macs(256, 256, k, HW, groups=grp)
    mem = 2 * 256 * HW * HW + conv_params(256, 256, k, grp)     # 读输入 + 写输出 + 权重
    ai = macs / mem
    rows.append((name, macs, mem, ai))
    print(f'{name:<26s} {macs:>14,d} {mem:>11,d} {ai:>10.1f} '
          f'{("计算受限" if ai > 100 else "**访存受限**"):>8s}')

ai = {r[0]: r[3] for r in rows}
assert ai['dense 3x3'] > 40 * ai['depthwise 5x5'], 'dense 与 depthwise 的算术强度差两个数量级'
assert ai['depthwise 5x5'] < 20
print()
print('⚠️  这是「FLOPs 骗人」的经典案例：depthwise 的 FLOPs 极低，')
print('    但墙钟时间由「读一遍输入、写一遍输出」决定 —— **而这部分与 k 无关**。')
print('    所以 3x3 -> 5x5 的延迟增量很小；但 k 继续增大时权重驻留、寄存器压力、')
print('    以及 kernel 优化不足会让延迟**非线性地跳上去**。')
print('✅ 正确说法：FLOPs 是计算受限算子的良好代理，对访存受限算子完全失效。')

In [ ]:
print('③ 固定参数预算，哪个核尺寸买到最大的 ERF？')
BUDGET, C = 2_000_000, 256
print(f'预算 {BUDGET:,} 参数，通道数 {C}，每层 = depthwise k*k + pointwise 1x1')
print()
print(f'{"k":>4s} {"每层参数":>10s} {"能堆几层":>9s} {"ERF sigma":>11s} {"TRF半径":>8s}')
best = None
for k in [3, 5, 7, 9, 11]:
    per = conv_params(C, C, k, groups=C) + conv_params(C, C, 1)
    n = BUDGET // per
    sig = erf_sigma_theory([(k, 1)] * n)
    print(f'{k:>4d} {per:>10,d} {n:>9d} {sig:>11.3f} {n * (k - 1) // 2:>8d}')
    if best is None or sig > best[1]:
        best = (k, sig)

sig3 = erf_sigma_theory([(3, 1)] * (BUDGET // (conv_params(C, C, 3, groups=C) + C * C)))
assert best[0] == 11, '纯参数量视角下，核越大 ERF 越大'
assert best[1] / sig3 > 3.0, 'k=11 的 ERF 应是 k=3 的 3 倍以上'
print()
print(f'✅ 纯参数量视角：核越大越划算（k=11 的 ERF 是 k=3 的 {best[1]/sig3:.1f} 倍）。')
print('⚠️  **所以「为什么 RTMDet 停在 5x5」的答案一定不在参数量里，而在上一格的访存账里。**')
print('   RTMDet 的消融结论与此一致：3x3->5x5 有明确 AP 提升且延迟几乎不变；')
print('   5x5->7x7 精度收益消失、延迟开始上升。5x5 是精度-延迟曲线的拐点。')

## 7 · 共享 conv + 每层独立 BN：参数量账与「为什么 BN 不能共享」

In [ ]:
def head_params(feat=256, stacked=2, n_levels=3, n_cls=80, mode='shared_conv_sep_bn'):
    '''三种检测头方案的参数量。cls/reg 两条支路各 stacked 层 3x3。'''
    conv = feat * feat * 9
    bn = 2 * feat                       # gamma + beta（running stats 不算可学参数）
    out = (feat * n_cls + n_cls) + (feat * 4 + 4)      # 每层各自的 1x1 输出头
    if mode == 'per_level':
        body = n_levels * stacked * 2 * (conv + bn)
    elif mode == 'fully_shared':
        body = stacked * 2 * (conv + bn)
    elif mode == 'shared_conv_sep_bn':
        body = stacked * 2 * conv + n_levels * stacked * 2 * bn
    else:
        raise ValueError(mode)
    return body + n_levels * out

MODES = [('per_level', '① 每层独立头'),
         ('fully_shared', '② 全共享（含 BN）'),
         ('shared_conv_sep_bn', '③ 共享 conv + 独立 BN')]
res = {mode: head_params(mode=mode) for mode, _ in MODES}
print(f'{"方案":<26s} {"参数量":>12s} {"相对最省":>10s}')
for mode, name in MODES:
    print(f'{name:<26s} {res[mode]:>12,d} {res[mode] / res["fully_shared"]:>9.3f}x')

assert res['per_level'] > 2.5 * res['shared_conv_sep_bn'], '独立头约为共享头的 3 倍'
assert res['shared_conv_sep_bn'] - res['fully_shared'] < 10_000, '独立 BN 只多几千个参数'
print()
print(f'✅ 方案③ 的 BN 参数是 3 层 x 2 支路 x 2 层 x 512 = {3*2*2*512:,}，方案② 只有 '
      f'2 支路 x 2 层 x 512 = {2*2*512:,}，')
print(f'   两者相差仅 {res["shared_conv_sep_bn"] - res["fully_shared"]:,} 个参数，'
      f'却拿回了正确的多尺度统计量。')
print(f'✅ 方案③ 比方案① 省 {1 - res["shared_conv_sep_bn"]/res["per_level"]:.0%} 的参数，')
print('   同时把三层的梯度汇到同一套权重上 —— **等价于把训练样本量乘了 3**。')

In [ ]:
# 为什么 BN 不能共享：FPN 三层的特征分布根本不是一个量级
rg = np.random.default_rng(0)
FEATS = {                                   # (格点数, 均值, 标准差) —— 高分辨率层幅值小、语义弱
    'P3 (s=8)':  rg.normal(0.2, 0.5, 4000),
    'P4 (s=16)': rg.normal(1.0, 1.5, 1000),
    'P5 (s=32)': rg.normal(2.5, 4.0, 250),
}
pooled = np.concatenate(list(FEATS.values()))
gm, gv = pooled.mean(), pooled.var()
print(f'三层混在一起的统计量: mean={gm:.3f}  var={gv:.3f}')
print(f'（注意格点数 4000:1000:250 —— 混合统计量会被 P3 主导）')
print()
print(f'{"层":<12s} {"原始 mean":>10s} {"原始 std":>9s} | {"共享BN后 mean":>14s} {"std":>7s} '
      f'| {"独立BN后 mean":>14s} {"std":>7s}')
worst = 0.0
for name, v in FEATS.items():
    z_shared = (v - gm) / np.sqrt(gv + 1e-5)
    z_sep = (v - v.mean()) / np.sqrt(v.var() + 1e-5)
    worst = max(worst, abs(z_shared.mean()))
    print(f'{name:<12s} {v.mean():>10.3f} {v.std():>9.3f} | {z_shared.mean():>+14.3f} '
          f'{z_shared.std():>7.3f} | {z_sep.mean():>+14.3f} {z_sep.std():>7.3f}')
    assert abs(z_sep.mean()) < 1e-9 and abs(z_sep.std() - 1.0) < 1e-4

assert worst > 1.0, '共享 BN 下至少有一层的归一化均值偏离 0 超过 1 个标准差'
print()
print(f'⚠️  共享 BN 下，P5 归一化后的均值是 {(FEATS["P5 (s=32)"].mean()-gm)/np.sqrt(gv+1e-5):+.2f}、'
      f'标准差 {FEATS["P5 (s=32)"].std()/np.sqrt(gv+1e-5):.2f} ——')
print('    这一层根本没有被正确归一化。而独立 BN 下三层都精确落在 (0, 1)。')
print('✅ 卷积权重该共享（「什么纹理是交通标志」三个尺度上是同一个函数），')
print('   BN 统计量必须独立（三层的特征分布差一个量级）。RTMDetSepBNHead 的全部设计就是这句话。')
print('✅ 部署副产品：推理时 BN 折叠进 conv -> 导出后变成「每层一套独立 conv 权重」，')
print('   训练享受共享的正则化，推理不承担任何额外开销（与模块 01 的重参数化同源）。')

## 8 · DynamicSoftLabelAssigner：完整代价矩阵

$$C_{ig} = \underbrace{\textstyle\sum_c \mathrm{BCE}(p_{ic}, y_{igc})\cdot|y_{igc}-p_{ic}|^2}_{\text{软标签分类代价}}
+ \underbrace{3\cdot(-\log u_{ig})}_{\text{IoU 代价}}
+ \underbrace{10^{\,d_{ig}/s_i-3}}_{\text{软中心先验}}$$

其中 $y_{igc}=u_{ig}\cdot\mathbf{1}[c=c_g]$ —— **分类目标不是 1，而是该格点预测框的 IoU**。

In [ ]:
# ---- 一个紧凑的合成场景（320x320，三层 FPN，3 个 GT）----
IMG, STRIDES = 320, [8, 16, 32]

def make_points(img=IMG, strides=STRIDES):
    P, S = [], []
    for s in strides:
        n = img // s
        gy, gx = np.meshgrid(np.arange(n), np.arange(n), indexing='ij')
        P.append(np.stack([(gx.ravel() + 0.5) * s, (gy.ravel() + 0.5) * s], 1).astype(float))
        S.append(np.full(n * n, float(s)))
    return np.concatenate(P), np.concatenate(S)

POINTS, PT_S = make_points()
GT = np.array([[150., 90., 168., 108.],      # 18x18 远处标志
               [ 60., 180., 108., 228.],     # 48x48
               [200., 160., 264., 224.]])    # 64x64
GT_CLS = np.array([0, 1, 0])
N_CLS = 3

def bbox_iou(a, b):
    aa = (a[:, 2] - a[:, 0]).clip(0) * (a[:, 3] - a[:, 1]).clip(0)
    ab = (b[:, 2] - b[:, 0]).clip(0) * (b[:, 3] - b[:, 1]).clip(0)
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = (rb - lt).clip(0)
    inter = wh[..., 0] * wh[..., 1]
    return inter / (aa[:, None] + ab[None, :] - inter + 1e-12)

def synth_detector(points, gt, gt_cls, n_cls=N_CLS, seed=5):
    rg = np.random.default_rng(seed)
    N = len(points)
    ctr = (gt[:, :2] + gt[:, 2:]) / 2
    sz = np.sqrt((gt[:, 2] - gt[:, 0]) * (gt[:, 3] - gt[:, 1]))
    d = np.linalg.norm(points[:, None, :] - ctr[None, :, :], axis=2) / sz[None, :]
    near, q = d.argmin(1), np.exp(-(d.min(1) / 0.7) ** 2)
    g = gt[near]
    gcx, gcy = (g[:, 0] + g[:, 2]) / 2, (g[:, 1] + g[:, 3]) / 2
    gw, gh = g[:, 2] - g[:, 0], g[:, 3] - g[:, 1]
    j, nz = 1 - q, rg.normal(size=(N, 4))
    pcx, pcy = gcx + nz[:, 0] * .45 * gw * j, gcy + nz[:, 1] * .45 * gh * j
    pw, ph = gw * np.exp(nz[:, 2] * .4 * j), gh * np.exp(nz[:, 3] * .4 * j)
    box = np.stack([pcx - pw / 2, pcy - ph / 2, pcx + pw / 2, pcy + ph / 2], 1)
    logit = rg.normal(-3.6, .6, size=(N, n_cls))
    logit[np.arange(N), gt_cls[near]] += 6.6 * q + rg.normal(0, .9, N)
    return 1 / (1 + np.exp(-logit)), box

CLS_PRED, BOX_PRED = synth_detector(POINTS, GT, GT_CLS)
print(f'格点 {len(POINTS)} 个（40^2+20^2+10^2）, GT {len(GT)} 个, 类别 {N_CLS}')
assert len(POINTS) == 2100 and CLS_PRED.shape == (2100, 3)
print('✅ 合成场景就位')

In [ ]:
EPS = 1e-7

def dsla_cost(points, pt_stride, gt, gt_cls, cls_pred, box_pred,
              iou_weight=3.0, soft_center_radius=3.0):
    '''RTMDet 的 DynamicSoftLabelAssigner 代价矩阵，返回 (G,N) 与三项分解。'''
    G, N, C = len(gt), len(points), cls_pred.shape[1]
    ious = bbox_iou(gt, box_pred).clip(0.0, 1.0)                   # (G,N)

    onehot = np.zeros((G, C)); onehot[np.arange(G), gt_cls] = 1.0
    soft = ious[:, :, None] * onehot[:, None, :]                   # (G,N,C) ← 软标签 = IoU
    p = np.clip(cls_pred, EPS, 1 - EPS)[None]                      # (1,N,C)
    bce = -(soft * np.log(p) + (1 - soft) * np.log(1 - p))
    cls_cost = (bce * (soft - p) ** 2).sum(-1)                     # ← |y-p|^2 调制

    iou_cost = -np.log(np.clip(ious, EPS, None)) * iou_weight

    gctr = (gt[:, :2] + gt[:, 2:]) / 2
    d = np.linalg.norm(points[None, :, :] - gctr[:, None, :], axis=2) / pt_stride[None, :]
    center_cost = 10.0 ** (d - soft_center_radius)

    return cls_cost + iou_cost + center_cost, dict(
        cls=cls_cost, iou=iou_cost, center=center_cost, ious=ious, soft=soft)

COST, PART = dsla_cost(POINTS, PT_S, GT, GT_CLS, CLS_PRED, BOX_PRED)
print(f'代价矩阵 shape = {COST.shape}')
print()
print(f'{"GT":>3s} {"尺寸":>7s} {"最小代价":>10s} {"其中 cls":>10s} {"iou":>9s} {"center":>9s}')
for g in range(len(GT)):
    i = COST[g].argmin()
    print(f'{g:>3d} {GT[g,2]-GT[g,0]:>4.0f}px {COST[g,i]:>10.4f} {PART["cls"][g,i]:>10.4f} '
          f'{PART["iou"][g,i]:>9.4f} {PART["center"][g,i]:>9.4f}')

# —— 性质 1：预测与软标签完全一致时，分类代价严格为 0 ——
g, i = 0, COST[0].argmin()
soft_gi = PART['soft'][g, i]
p_perfect = np.clip(soft_gi, EPS, 1 - EPS)[None, :]
cls_perfect = (-(soft_gi * np.log(p_perfect) + (1 - soft_gi) * np.log(1 - p_perfect))
               * (soft_gi - p_perfect) ** 2).sum()
print()
print(f'软标签 y = {soft_gi}  (IoU={PART["ious"][g,i]:.4f} 放在 GT 类别那一维)')
print(f'若预测恰好 p = y，分类代价 = {cls_perfect:.3e}')
assert cls_perfect < 1e-12, '|y-p|^2 调制因子应让完全对齐的格点代价归零'

# —— 性质 2：软中心先验的三个刻度 ——
for dn, expect in [(0.0, 1e-3), (3.0, 1.0), (5.0, 100.0)]:
    assert abs(10.0 ** (dn - 3.0) - expect) < 1e-9
print(f'软中心先验刻度: d/s=0 -> {10.0**-3:.4f}   d/s=3 -> {10.0**0:.1f}   d/s=5 -> {10.0**2:.0f}')

# —— 性质 3：IoU 代价随 IoU 单调减 ——
u = np.array([0.1, 0.3, 0.5, 0.7, 0.9])
assert np.all(np.diff(-np.log(u) * 3.0) < 0)
print(f'IoU 代价 (-3·log u): ' + '  '.join(f'u={a:.1f}->{-np.log(a)*3:.2f}' for a in u))
print()
print('✅ 三项各司其职：软标签把「框准不准」写进分类目标；|y-p|^2 让已对齐的格点退出争抢；')
print('   IoU 代价排除远框；软中心先验用**斜坡**替代 SimOTA 的 0/100000 **悬崖**。')

In [ ]:
# 剩下两步与 SimOTA 相同：dynamic-k（RTMDet 用 top-13）+ 去冲突（代价最小者胜）
def dsla_assign(cost, ious, topk=13):
    G, N = cost.shape
    nk = min(topk, N)
    dyn_k = np.clip(np.floor(np.sort(ious, 1)[:, -nk:].sum(1)).astype(int), 1, None)
    matching = np.zeros((G, N), bool)
    for g in range(G):
        matching[g, np.argsort(cost[g])[:dyn_k[g]]] = True
    multi = matching.sum(0) > 1
    n_conf = int(multi.sum())
    if multi.any():
        win = np.where(matching, cost, np.inf)[:, multi].argmin(0)
        matching[:, multi] = False
        matching[win, np.where(multi)[0]] = True
    assign = np.full(N, -1)
    has = matching.any(0)
    assign[has] = matching[:, has].argmax(0)
    return assign, dyn_k, n_conf

ASSIGN, DYN_K, NCONF = dsla_assign(COST, PART['ious'])
gctr = (GT[:, :2] + GT[:, 2:]) / 2
dnorm = np.linalg.norm(POINTS[None] - gctr[:, None], axis=2) / PT_S[None]

print(f'{"GT":>3s} {"尺寸":>7s} {"dynamic-k":>10s} {"正样本":>7s} {"正样本平均IoU":>14s} '
      f'{"平均 d/stride":>14s}')
for g in range(len(GT)):
    m = ASSIGN == g
    print(f'{g:>3d} {GT[g,2]-GT[g,0]:>4.0f}px {DYN_K[g]:>10d} {int(m.sum()):>7d} '
          f'{PART["ious"][g, m].mean():>14.3f} {dnorm[g, m].mean():>14.3f}')
print(f'去冲突仲裁次数: {NCONF}')

assert DYN_K.min() >= 1 and (ASSIGN >= 0).sum() == DYN_K.sum() - NCONF
assert all(dnorm[g, ASSIGN == g].mean() < 3.0 for g in range(len(GT))),     '软中心先验应把正样本压在 3 个 stride 以内'
assert all(PART['ious'][g, ASSIGN == g].mean() > 0.5 for g in range(len(GT)))
print()
print('✅ 所有正样本的平均中心距离都在 3 个 stride 以内 —— 软先验在 d/s=3 处代价已经等于 1，')
print('   足以压过大多数格点的 cls+iou 优势。这就是「斜坡」的作用方式。')
print('✅ 一句话总结 DSLA：**把 SimOTA 的硬标签换成 IoU 软标签，把 0/100000 硬先验')
print('   换成 10^(d/s-3) 软先验，其余（dynamic-k、去冲突）完全不变。**')

## 9 · 模型缩放计算器：tiny → x

`deepen_factor` 控制每个 stage 的 block 数，`widen_factor` 控制通道数。
下面这个计算器是**简化的结构估算**（略去 SPPF、通道注意力等），
但我们会验证：**它对官方数字的比值在五个档位上几乎恒定** —— 说明缩放趋势被完整保留。

In [ ]:
def cp(cin, cout, k, groups=1, bn=True):
    return (cin // groups) * cout * k * k + (2 * cout if bn else 0)

def cf(cin, cout, k, hw, groups=1):
    return (cin // groups) * cout * k * k * hw * hw

def cspnext_block(c):                       # 3x3 + 5x5 DW + 1x1
    return cp(c, c, 3) + cp(c, c, 5, groups=c) + cp(c, c, 1)

def cspnext_block_f(c, hw):
    return cf(c, c, 3, hw) + cf(c, c, 5, hw, groups=c) + cf(c, c, 1, hw)

def csp_layer(cin, cout, n):                # CSP: 主/短两支 1x1 + n 个 block + 汇合 1x1
    mid = cout // 2
    return cp(cin, mid, 1) * 2 + cp(mid * 2, cout, 1) + n * cspnext_block(mid)

def csp_layer_f(cin, cout, n, hw):
    mid = cout // 2
    return cf(cin, mid, 1, hw) * 2 + cf(mid * 2, cout, 1, hw) + n * cspnext_block_f(mid, hw)

def rtmdet_stats(widen=1.0, deepen=1.0, res=640, n_cls=80, n_lv=3):
    ch = [int(64 * widen), int(128 * widen), int(256 * widen),
          int(512 * widen), int(1024 * widen)]
    nb = [max(round(3 * deepen), 1), max(round(6 * deepen), 1),
          max(round(6 * deepen), 1), max(round(3 * deepen), 1)]
    Pm = Fm = 0
    hw = res // 2
    Pm += cp(3, ch[0] // 2, 3) + cp(ch[0] // 2, ch[0] // 2, 3) + cp(ch[0] // 2, ch[0], 3)
    Fm += cf(3, ch[0] // 2, 3, hw) + cf(ch[0] // 2, ch[0] // 2, 3, hw) + cf(ch[0] // 2, ch[0], 3, hw)
    for i in range(4):                                    # backbone 4 个 stage
        hw //= 2
        Pm += cp(ch[i], ch[i + 1], 3);            Fm += cf(ch[i], ch[i + 1], 3, hw)
        Pm += csp_layer(ch[i + 1], ch[i + 1], nb[i]); Fm += csp_layer_f(ch[i + 1], ch[i + 1], nb[i], hw)
    nch, nn = ch[2], max(round(3 * deepen), 1)            # neck: PAFPN，统一到 ch[2]
    sizes = [res // 8, res // 16, res // 32]
    for i, s in enumerate(sizes):
        Pm += cp([ch[2], ch[3], ch[4]][i], nch, 1); Fm += cf([ch[2], ch[3], ch[4]][i], nch, 1, s)
    for s in sizes[:2]:
        Pm += csp_layer(nch * 2, nch, nn); Fm += csp_layer_f(nch * 2, nch, nn, s)
    for s in sizes[1:]:
        Pm += cp(nch, nch, 3); Fm += cf(nch, nch, 3, s)
        Pm += csp_layer(nch * 2, nch, nn); Fm += csp_layer_f(nch * 2, nch, nn, s)
    for _ in range(2):                                    # head: 共享 conv + 每层独立 BN
        Pm += cp(nch, nch, 3, bn=False) * 2 + 2 * 2 * nch * n_lv
        for s in sizes:
            Fm += cf(nch, nch, 3, s) * 2
    Pm += (nch * n_cls + n_cls + nch * 4 + 4) * n_lv
    for s in sizes:
        Fm += (nch * n_cls + nch * 4) * s * s
    return Pm, Fm

OFFICIAL = {'tiny': (4.8, 8.1), 's': (8.89, 14.8), 'm': (24.71, 39.27),
            'l': (52.3, 80.23), 'x': (94.86, 141.67)}
VARIANTS = [('tiny', .375, .167), ('s', .5, .33), ('m', .75, .67),
            ('l', 1., 1.), ('x', 1.25, 1.33)]

print(f'{"档位":<6s} {"widen":>6s} {"deepen":>7s} {"估算参数":>10s} {"官方":>8s} {"比值":>7s} '
      f'{"估算MACs":>10s} {"官方":>8s} {"比值":>7s}')
pr, fr = [], []
for name, w, d in VARIANTS:
    p, f = rtmdet_stats(w, d)
    op, of = OFFICIAL[name]
    pr.append(p / 1e6 / op); fr.append(f / 1e9 / of)
    print(f'{name:<6s} {w:>6.3f} {d:>7.3f} {p/1e6:>9.2f}M {op:>7.2f}M {pr[-1]:>7.3f} '
          f'{f/1e9:>9.2f}G {of:>7.2f}G {fr[-1]:>7.3f}')

print()
print(f'参数比值区间 [{min(pr):.3f}, {max(pr):.3f}]，极差 {max(pr)/min(pr):.3f}x')
print(f'MACs 比值区间 [{min(fr):.3f}, {max(fr):.3f}]，极差 {max(fr)/min(fr):.3f}x')
assert max(pr) / min(pr) < 1.05, '简化估算器对官方参数量的比值应在五档上几乎恒定'
assert max(fr) / min(fr) < 1.05
print('✅ 简化估算器系统性偏离官方值（参数约 0.55x、MACs 约 0.77x，因为略去了 SPPF/注意力等），')
print('   但**比值在五个档位上几乎恒定** —— 缩放趋势被完整保留，这正是我们要验证的东西。')

In [ ]:
# 三条缩放规律，各 assert 一遍
p1, f1 = rtmdet_stats(1.0, 1.0, 640)
p2, _ = rtmdet_stats(2.0, 1.0, 640)
p3, _ = rtmdet_stats(1.0, 2.0, 640)
_, f2 = rtmdet_stats(1.0, 1.0, 1280)

print(f'{"变化":<26s} {"参数量比":>10s} {"FLOPs比":>10s} {"理论":>14s}')
print(f'{"widen x2 (宽度翻倍)":<26s} {p2/p1:>10.3f} {"-":>10s} {"w^2 = 4.00":>14s}')
print(f'{"deepen x2 (深度翻倍)":<26s} {p3/p1:>10.3f} {"-":>10s} {"近似线性":>14s}')
print(f'{"res x2 (分辨率翻倍)":<26s} {1.000:>10.3f} {f2/f1:>10.3f} {"r^2 = 4.00":>14s}')

assert 3.9 < p2 / p1 < 4.1, '参数量对宽度是平方关系'
assert 1.3 < p3 / p1 < 1.8, '参数量对深度近似线性（stem/neck/head 不随深度变）'
assert abs(f2 / f1 - 4.0) < 1e-9, 'FLOPs 对分辨率精确是平方关系'
assert rtmdet_stats(1., 1., 1280)[0] == p1, '分辨率完全不影响参数量'
print()
print('✅ 三条规律：参数 ∝ w^2 · f(d)，与分辨率无关；FLOPs ∝ w^2 · f(d) · r^2。')
print('⚠️  为什么 RTMDet 的宽深比这么保守（x 档只有 widen=1.25）？')
print('    **宽度带来并行度，深度带来串行延迟。**')
print('    加宽通常能被 GPU/NPU 的并行度吃掉，延迟增长远小于 FLOPs 增长；')
print('    加深则每层都要等前一层算完，延迟是硬加上去的。')
print('    -> 面向延迟优化的模型族倾向「宽而浅」，面向 FLOPs 的倾向「窄而深」。')
print()
print('TSR 选型的实用法则：先定延迟预算，再在帕累托前沿取点。')
print('  · 车端 33ms 里 TSR 常常只分到 3~5ms -> 基本框定在 tiny/s，或 m + INT8')
print('  · x 档的正确用法不是上车，是**当云端教师**：跑回传数据做自动标注与难例挖掘')

## ✏️ 练习 1：通用理论感受野计算器

实现 `theoretical_rf(specs)`，`specs = [(k, stride, dilation), ...]`（从输入侧往输出侧）。
返回 `(rf_diameter, total_stride)`。

递推：$r \mathrel{+}= (k-1)\cdot d\cdot j$；$j \mathrel{*}= s$；初值 $r=1,\ j=1$。

In [ ]:
def theoretical_rf(specs):
    # TODO: 返回 (感受野直径, 总 stride)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert theoretical_rf([(3, 1, 1)]) == (3, 1)
assert theoretical_rf([(5, 1, 1)]) == (5, 1)
assert theoretical_rf([(3, 1, 2)]) == (5, 1), '3x3 空洞率2 的 TRF 等于 5x5'
assert theoretical_rf([(3, 1, 1)] * 10) == (21, 1)
assert theoretical_rf([(3, 1, 1)] * 2) == theoretical_rf([(5, 1, 1)]), '两个3x3 == 一个5x5'
assert theoretical_rf([(1, 1, 1)] * 5) == (1, 1), '1x1 卷积不扩大感受野'
# 下采样让后续每一层的贡献都乘以累计 stride
assert theoretical_rf([(3, 2, 1), (3, 1, 1)]) == (7, 2)
assert theoretical_rf([(3, 2, 1), (3, 2, 1), (3, 1, 1)]) == (15, 4)

print(f'{"结构":<44s} {"TRF直径":>8s} {"总stride":>9s}')
STACKS = [
    ('单层 3x3',                          [(3, 1, 1)]),
    ('CSPNeXtBlock (3x3 + 5x5DW + 1x1)',  [(3, 1, 1), (5, 1, 1), (1, 1, 1)]),
    ('同上但用 3x3 替代 5x5',             [(3, 1, 1), (3, 1, 1), (1, 1, 1)]),
    ('stem + 4 stage 下采样 (每 stage 1 block)',
     [(3, 2, 1), (3, 2, 1), (3, 1, 1), (5, 1, 1), (3, 2, 1), (3, 1, 1), (5, 1, 1),
      (3, 2, 1), (3, 1, 1), (5, 1, 1), (3, 2, 1), (3, 1, 1), (5, 1, 1)]),
]
for name, sp in STACKS:
    r, j = theoretical_rf(sp)
    print(f'{name:<44s} {r:>8d} {j:>9d}')

r_big, j_big = theoretical_rf(STACKS[-1][1])
assert j_big == 32 and r_big > 300
print()
print(f'✅ 练习 1 通过：到 stride 32 那一层，TRF 直径已经有 {r_big} 像素 —— 远超 640 的输入。')
print('   **TRF 早就饱和了，但精度还在随大核提升 —— 这正是必须区分 TRF 与 ERF 的原因。**')

## ✏️ 练习 2：ERF 的解析预测器，并与数值实验对拍

实现 `erf_sigma(specs)`，`specs = [(k, dilation), ...]`（stride=1）：
$\sigma = \sqrt{\sum_\ell d_\ell^2 (k_\ell^2-1)/12}$。

再实现 `erf_over_trf(k, n)`：同核堆叠 $n$ 层时 ERF 标准差与 TRF 半径的比值。

In [ ]:
def erf_sigma(specs):
    # TODO
    raise NotImplementedError

def erf_over_trf(k, n):
    # TODO: erf_sigma([(k,1)]*n) / (n*(k-1)/2)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert abs(erf_sigma([(3, 1)]) - np.sqrt(8 / 12)) < 1e-12
assert abs(erf_sigma([(5, 1)]) - np.sqrt(2.0)) < 1e-12
assert abs(erf_sigma([(1, 1)] * 9) - 0.0) < 1e-12, '1x1 卷积不扩大 ERF'
assert abs(erf_sigma([(3, 2)]) - erf_sigma([(3, 1)]) * 2) < 1e-12, '空洞率让 sigma 线性放大'
assert abs(erf_sigma([(3, 1)] * 4) - erf_sigma([(3, 1)]) * 2) < 1e-12, '层数 x4 -> sigma x2'

# 与第 2 节的梯度回传数值实验对拍（这是本练习的重点）
for sp in [[(3, 1)] * 6, [(5, 1)] * 4, [(7, 3)], [(3, 1), (5, 1), (3, 2), (5, 1)]]:
    meas = spatial_std(erf_map(sp))
    assert abs(meas - erf_sigma(sp)) < 1e-9, (sp, meas, erf_sigma(sp))
print('✅ 与梯度回传数值实验逐项对拍通过（误差 < 1e-9）')

assert abs(erf_over_trf(3, 1) - np.sqrt(8 / 12) / 1.0) < 1e-12
assert abs(erf_over_trf(3, 4) / erf_over_trf(3, 64) - 4.0) < 1e-9, 'ERF/TRF 按 1/sqrt(N) 衰减'
print()
print(f'{"k":>4s} ' + ' '.join(f'{f"N={n}":>10s}' for n in [1, 4, 16, 64]))
for k in [3, 5, 7]:
    print(f'{k:>4d} ' + ' '.join(f'{erf_over_trf(k, n):>10.4f}' for n in [1, 4, 16, 64]))
print()
print('✅ 练习 2 通过。反过来用这个公式做**结构设计**：')
print('   给定目标 sigma，可以直接反解「用什么核、堆几层」——')
print('   例如想要 sigma=8，5x5 需要 n = 8^2*12/24 = 32 层，7x7 只要 16 层。')
n5 = 8 ** 2 * 12 / (5 ** 2 - 1)
n7 = 8 ** 2 * 12 / (7 ** 2 - 1)
assert abs(n5 - 32) < 1e-9 and abs(n7 - 16) < 1e-9
print(f'   验算: 5x5 需 {n5:.0f} 层, 7x7 需 {n7:.0f} 层')

## ✏️ 练习 3：固定参数预算下的最优核尺寸

实现 `best_kernel(budget, C, kernels)`：每层是「$k\times k$ depthwise + $1\times1$ pointwise」，
参数量 $= Ck^2 + C^2$。在预算内能堆 $n=\lfloor budget/per\rfloor$ 层，ERF 为 $\sqrt{n(k^2-1)/12}$。

返回 `(best_k, {k: (per_layer, n_layers, sigma)})`。

In [ ]:
def best_kernel(budget, C, kernels=(3, 5, 7, 9, 11)):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
bk, table = best_kernel(2_000_000, 256)
assert set(table) == {3, 5, 7, 9, 11}
assert table[3][0] == 256 * 9 + 256 * 256 == 67840
assert table[3][1] == 2_000_000 // 67840 == 29
assert abs(table[3][2] - np.sqrt(29 * 8 / 12)) < 1e-9
assert bk == 11, '纯参数量视角下核越大越划算'
assert table[11][2] / table[3][2] > 3.0

print(f'预算 2,000,000 参数, C=256')
print(f'{"k":>4s} {"每层参数":>10s} {"层数":>6s} {"ERF sigma":>11s} {"相对k=3":>9s}')
for k in sorted(table):
    per, n, sig = table[k]
    print(f'{k:>4d} {per:>10,d} {n:>6d} {sig:>11.3f} {sig/table[3][2]:>8.2f}x')

# 通道数越大，空间核的相对成本越低 -> 大核越划算
bk_small, t_small = best_kernel(2_000_000, 64)
print()
print(f'C=64  时最优 k = {bk_small};  C=256 时最优 k = {bk}')
assert t_small[11][2] / t_small[3][2] < table[11][2] / table[3][2],     'C 越小，pointwise 占比越低，大核的相对优势越小'
print()
print('✅ 练习 3 通过。**但现实里 RTMDet 停在 5x5** ——')
print('   因为 depthwise 是访存受限的：墙钟时间由「读输入+写输出」决定，与 k 基本无关，')
print('   而 k 大到 9/11 时权重驻留与 kernel 优化不足会让延迟非线性跳升。')
print('   **参数量的账说「上」，硬件的账说「停」。这就是 5x5 这个数字的来历。**')

## ✏️ 练习 4：Quality Focal Loss —— 软标签必须配的损失

$\mathrm{QFL}(p, y) = |y-p|^{\beta}\cdot\big[-(y\log p + (1-y)\log(1-p))\big]$，默认 $\beta=2$。

实现 `quality_focal_loss(p, y, beta=2.0)`（逐元素，返回同形状数组）。
注意数值稳定：把 `p` clip 到 `[1e-7, 1-1e-7]`。

In [ ]:
def quality_focal_loss(p, y, beta=2.0):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
p = np.array([0.1, 0.5, 0.8, 0.8, 0.2])
y = np.array([0.0, 0.5, 0.8, 0.2, 0.9])
L = quality_focal_loss(p, y)
assert L.shape == p.shape

# 性质 ①：预测恰好等于目标 -> 损失为 0（|y-p|^beta = 0）
assert L[1] < 1e-12 and L[2] < 1e-12, '完全对齐时 QFL = 0'
# 性质 ②：偏离越大损失越大（同一个 y 下单调）
grid = np.linspace(0.01, 0.99, 99)
lv = quality_focal_loss(grid, np.full_like(grid, 0.7))
i0 = int(np.argmin(np.abs(grid - 0.7)))
assert lv[i0] == lv.min(), 'QFL 在 p=y 处取最小'
assert np.all(np.diff(lv[:i0]) < 0) and np.all(np.diff(lv[i0:]) > 0), 'p=y 两侧单调'
# 性质 ③：y 为 0/1 时退化成 focal 风格（易样本被强烈降权）
easy = quality_focal_loss(np.array([0.99]), np.array([1.0]))[0]
hard = quality_focal_loss(np.array([0.10]), np.array([1.0]))[0]
assert hard > 500 * easy, '难样本的权重应远大于易样本'
# 性质 ④：beta 越大，易样本被压得越狠
assert (quality_focal_loss(np.array([0.9]), np.array([1.0]), beta=4.0)[0]
        < quality_focal_loss(np.array([0.9]), np.array([1.0]), beta=2.0)[0])

print(f'{"p":>6s} {"y":>6s} {"|y-p|^2":>9s} {"BCE":>9s} {"QFL":>10s}')
for pi, yi in zip(p, y):
    pc = np.clip(pi, 1e-7, 1 - 1e-7)
    bce = -(yi * np.log(pc) + (1 - yi) * np.log(1 - pc))
    print(f'{pi:>6.2f} {yi:>6.2f} {abs(yi-pi)**2:>9.4f} {bce:>9.4f} '
          f'{quality_focal_loss(np.array([pi]), np.array([yi]))[0]:>10.5f}')
print()
print('✅ 练习 4 通过。**为什么必须换损失**：普通 FocalLoss 只接受 0/1 标签，')
print('   喂给它一个连续目标值会**静默出错**（不报错、AP 掉几个点）。')
print('   QFL 把 focal 的调制项从「离散的难易」推广到「连续的偏离量」，')
print('   于是同一套机制既能处理硬标签也能处理软标签。')
print('⚠️  连带后果：软标签让分类分数整体下移（目标值从 1.0 变成 IoU≈0.6~0.9）——')
print('   **换分配器后 score 阈值必须在验证集上重扫**，否则召回会莫名掉一截。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def theoretical_rf(specs):
    r, j = 1, 1
    for (k, s, d) in specs:
        r += (k - 1) * d * j
        j *= s
    return r, j

In [ ]:
# 练习 2 参考答案
def erf_sigma(specs):
    return float(np.sqrt(sum(d * d * (k * k - 1) / 12 for k, d in specs)))

def erf_over_trf(k, n):
    return erf_sigma([(k, 1)] * n) / (n * (k - 1) / 2)

In [ ]:
# 练习 3 参考答案
def best_kernel(budget, C, kernels=(3, 5, 7, 9, 11)):
    table = {}
    for k in kernels:
        per = C * k * k + C * C          # depthwise k x k + pointwise 1x1
        n = budget // per
        table[k] = (per, int(n), float(np.sqrt(n * (k * k - 1) / 12)))
    return max(table, key=lambda k: table[k][2]), table

In [ ]:
# 练习 4 参考答案
def quality_focal_loss(p, y, beta=2.0):
    p = np.clip(np.asarray(p, dtype=float), 1e-7, 1 - 1e-7)
    y = np.asarray(y, dtype=float)
    bce = -(y * np.log(p) + (1 - y) * np.log(1 - p))
    return np.abs(y - p) ** beta * bce

---
## 🧪 真实工程胶囊：RTMDet 的关键配置与「照抄哪些、别抄哪些」

In [ ]:
RECIPE = r'''
# ============ ① mmdetection 里 RTMDet 的关键配置块 ============
model = dict(
    type='RTMDet',
    backbone=dict(
        type='CSPNeXt', arch='P5',
        deepen_factor=0.67, widen_factor=0.75,      # ← m 档；五档只改这两个数
        channel_attention=True,                      # 小模型收益明显，大模型递减
        norm_cfg=dict(type='SyncBN'), act_cfg=dict(type='SiLU', inplace=True)),
    neck=dict(
        type='CSPNeXtPAFPN',
        in_channels=[192, 384, 768], out_channels=192,
        num_csp_blocks=2,                            # ← neck 与 backbone 用**同一种 block**
        expand_ratio=0.5),
    bbox_head=dict(
        type='RTMDetSepBNHead',
        in_channels=192, feat_channels=192, stacked_convs=2,
        share_conv=True,                             # ← **卷积权重跨层共享，BN 每层独立**
        pred_kernel_size=1,
        loss_cls=dict(type='QualityFocalLoss', use_sigmoid=True, beta=2.0,
                      loss_weight=1.0),              # ← 软标签**必须**配 QFL
        loss_bbox=dict(type='GIoULoss', loss_weight=2.0)),
    train_cfg=dict(assigner=dict(
        type='DynamicSoftLabelAssigner',
        topk=13,                                     # dynamic-k：top-13 IoU 求和
        iou_weight=3.0,
        soft_center_radius=3.0)),                    # 10^(d/stride - 3)
)

# ============ ② TSR 场景要改的地方（别原样照抄 COCO 配置） ============
# 1. soft_center_radius: 3.0 -> 2.5
#    密集小目标（主牌+辅助牌）候选池重叠严重，收紧中心先验能减少名额争夺。
#    代价：极小目标可用候选进一步减少 —— 必须配合下面第 2 条一起改。
# 2. 加 P2 层（stride 4）或提高输入分辨率。
#    16x16 的标志在 stride 8/16/32 上一共只有 2 个格点中心落在框内（模块 02 实测）。
#    **这是物理上限，换任何分配器都突破不了。**
# 3. score 阈值必须重扫。软标签把分类分数整体下移，沿用旧阈值会掉召回。
# 4. num_classes 大幅增加时，head 的 1x1 输出层参数会线性增长；
#    类别极多（>200）时考虑两级方案（类别无关检测 + 高分辨率 crop 分类，见 C55 m02）。

# ============ ③ 5x5 大核相关的部署检查 ============
# - depthwise 5x5 在 TensorRT 上是否被高效实现？用 trtexec --dumpProfile 看逐层耗时，
#   如果 DW 层占比异常高，说明 kernel 走了慢路径（换 TRT 版本或退回 3x3 验证）。
# - depthwise 层是**访存受限**的：FLOPs 低不等于快。别用 FLOPs 估延迟。
# - Conv+BN 折叠后，「共享 conv + 每层独立 BN」会展开成每层独立的 conv 权重，
#   engine 体积按层数增长 —— 导出后核对参数量，别以为共享头就一定省 engine。

# ============ ④ 换核尺寸/换分配器时的最小验证集 ============
#   [ ] 参数量与 MACs 变化（本 notebook 的计算器）
#   [ ] 目标硬件上的 p50/p99 延迟（不是 FLOPs！）
#   [ ] 分尺寸桶的 AP（小目标桶单独看，整体 mAP 会掩盖它）
#   [ ] 重扫 score 阈值后的 PR 曲线，而不是固定阈值下的单点
'''
print(RECIPE)
for key in ['CSPNeXt', 'deepen_factor', 'CSPNeXtPAFPN', 'RTMDetSepBNHead', 'share_conv',
            'QualityFocalLoss', 'DynamicSoftLabelAssigner', 'soft_center_radius',
            '访存受限', 'p99']:
    assert key in RECIPE, key
print('✅ 配方覆盖：五档同构配置 / TSR 定制改动 / 大核部署检查 / 最小验证集')

### 小结

- **理论感受野是几何量，有效感受野是统计量，精度跟着后者走。**
  TRF 半径按 $N$ 线性增长，ERF 标准差按 $\sqrt{N}$ 增长 ——
  两者之比按 $1/\sqrt{N}$ 衰减（3×3 堆 64 层时只剩 10%）。
- **$\sigma_{ERF}=\sqrt{\sum d_\ell^2(k_\ell^2-1)/12}$ 在均匀核线性网络下是恒等式**，
  本 notebook 用梯度回传法验证到小数点后 8 位。ReLU 门控会让真实 ERF 再小一圈
  （有效面积 205 → 52），但训练后的 ERF 又会显著大于初始化时的。
- **两个 3×3 与一个 5×5 的 TRF 完全相同，ERF 差 $\sqrt{1.5}$ 倍**；
  同深度下 5×5 相对 3×3 的 ERF 比值恒为 $\sqrt{3}$，与深度无关。
  **加大单层核比多堆几层划算，而且不增加串行延迟。**
- **可分离结构里成本几乎全在 1×1 pointwise 上**（C=256 时 5×5 DW 只占 8.9%），
  所以 3×3→5×5 只涨 5.7% 参数。**但 depthwise 是访存受限的** ——
  参数量的账说「核越大越好」，硬件的账说「停在 5×5」。
  **「FLOPs 是计算受限算子的良好代理，对访存受限算子完全失效」是面试高频点。**
- **共享 conv + 每层独立 BN**：省 67% 参数、把三层梯度汇到同一套权重（等价样本量 ×3），
  又不让 FPN 三层差一个量级的统计量互相污染。推理时 BN 折叠 → 零额外开销。
- **DSLA = SimOTA 换两件事**：硬标签 → IoU 软标签（分数携带定位质量，NMS 排序才对）；
  0/100000 硬先验 → $10^{d/s-3}$ 软先验（斜坡替代悬崖）。
  **软标签必须配 Quality Focal Loss，且 score 阈值必须重扫。**
- **缩放三规律**：参数 $\propto w^2 f(d)$、与分辨率无关；FLOPs $\propto w^2 f(d) r^2$。
  **宽度换并行度，深度换串行延迟** —— 面向延迟的模型族倾向「宽而浅」。
  x 档的正确用法是当云端教师做自动标注，不是上车。

下一站：**模块 04 · RT-DETR 解剖** —— 当 NMS 成为延迟方差的来源时，检测器该怎么重构。